# Score Based Model 2d gauss

## モジュールの呼び出し

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.preprocessing import StandardScaler

## ノイズスケジュールの定義

In [ ]:
# ハイパーパラメータ (必要に応じて調整)
timesteps = 500
beta_start = 0.0001
beta_end = 0.02
betas = torch.linspace(beta_start, beta_end, timesteps)
alphas = 1. - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)

def noise_schedule(t):
    sqrt_alpha_cumprod_t = torch.sqrt(alpha_cumprod[t])
    sqrt_one_minus_alpha_cumprod_t = torch.sqrt(1 - alpha_cumprod[t])
    return sqrt_alpha_cumprod_t, sqrt_one_minus_alpha_cumprod_t

## スコアネットワークの定義

In [ ]:
class ScoreNet2D(nn.Module):
    def __init__(self, input_dim, hidden_dim, time_embed_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim + time_embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)
        self.time_embed = nn.Linear(1, time_embed_dim)

    def forward(self, x, t):
        t_embed = self.time_embed(t[:, None].float() / timesteps)
        h = torch.cat([x, t_embed], dim=1)
        h = F.relu(self.fc1(h))
        h = F.relu(self.fc2(h))
        return self.fc3(h)

## 損失関数の定義

In [ ]:
def loss_fn(model, x_0, t):
    sqrt_alpha_cumprod_t, sqrt_one_minus_alpha_cumprod_t = noise_schedule(t)
    noise = torch.randn_like(x_0)
    x_t = sqrt_alpha_cumprod_t[:, None] * x_0 + sqrt_one_minus_alpha_cumprod_t[:, None] * noise
    predicted_score = model(x_t, t)
    target_score = -noise
    loss = F.mse_loss(predicted_score, target_score)
    return loss

## サンプル生成と2次元正規分布からの乱数データ(学習データ)生成

In [ ]:
@torch.no_grad()
def sample_2d(model, n_samples, input_dim, timesteps, alphas, alpha_cumprod, betas, device="cpu"):
    x_t = torch.randn(n_samples, input_dim).to(device)
    for i in reversed(range(timesteps)):
        t = torch.ones(n_samples, dtype=torch.long).to(device) * i
        sqrt_alpha_t = torch.sqrt(alphas[i])
        beta_t = betas[i]
        score_t = model(x_t, t)
        x_t = (1 / sqrt_alpha_t) * (x_t - (beta_t / torch.sqrt(1 - alpha_cumprod[i])) * score_t)
        if i > 0:
            noise = torch.randn_like(x_t)
            posterior_variance = beta_t
            x_t = x_t + torch.sqrt(posterior_variance) * noise
    return x_t.cpu().numpy()

# 2次元正規分布からのデータ生成
def generate_data_2d(n_samples=1000):
    mean = np.array([5, 5])
    cov = np.array([[1, 0.3], [0.3, 1]])
    return torch.tensor(np.random.multivariate_normal(mean, cov, n_samples), dtype=torch.float32)

In [ ]:
# シードの設定
torch.manual_seed(42)
np.random.seed(42)
# データの準備
data_2d = generate_data_2d(500)
# データの標準化
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_2d)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PyTorchテンソルへ明示的に float32 で変換
train_data = torch.tensor(scaled_data, dtype=torch.float32).to(device)
dataset_2d = TensorDataset(train_data)
dataloader_2d = DataLoader(dataset_2d, batch_size=64, shuffle=True)

# モデルとオプティマイザの初期化
input_dim = 2
hidden_dim = 128
time_embed_dim = 32
model_2d = ScoreNet2D(input_dim, hidden_dim, time_embed_dim)
optimizer_2d = Adam(model_2d.parameters(), lr=1e-3)


loss_history_2d = []
# 学習ループ (エポック数は適宜調整)
epochs = 1000
for epoch in range(epochs):
    for batch in dataloader_2d:
        x_0 = batch[0]
        t = torch.randint(0, timesteps, (x_0.shape[0],))
        loss = loss_fn(model_2d, x_0, t)
        optimizer_2d.zero_grad()
        loss.backward()
        optimizer_2d.step()
    loss_history_2d.append(loss.item())
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

## data_2d　可視化

In [ ]:
# data_2d　可視化
plt.figure(figsize=(8, 6))
plt.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.5, label='Data')
plt.title('2D Data Distribution')
plt.xlabel('X1')
plt.ylabel('X2')
plt.legend()
plt.grid()
plt.show()

## 損失のプロット

In [ ]:
# 損失のプロット
plt.plot(loss_history_2d, label='Training Loss')
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## サンプリングの実行

In [ ]:
# サンプリング
model_2d.eval()
sampled_data_2d = sample_2d(model_2d, n_samples=500, input_dim=input_dim, timesteps=timesteps, alphas=alphas, alpha_cumprod=alpha_cumprod, betas=betas)

## プロット範囲の設定

In [ ]:
# xとy 最大最小の計算してプロットの範囲を設定
data_2d_x_min_val = min(data_2d[:, 0])
data_2d_x_max_val = max(data_2d[:, 0])
data_2d_y_min_val = min(data_2d[:, 1])
data_2d_y_max_val = max(data_2d[:, 1])
data_2d_plt_lim = max(abs(data_2d_x_min_val), abs(data_2d_x_max_val), abs(data_2d_y_min_val), abs(data_2d_y_max_val))

x_min_val = np.min(sampled_data_2d[:, 0])
x_max_val = np.max(sampled_data_2d[:, 0])
y_min_val = np.min(sampled_data_2d[:, 1])
y_max_val = np.max(sampled_data_2d[:, 1])
sampled_data_plt_lim = max(abs(x_min_val), abs(x_max_val), abs(y_min_val), abs(y_max_val))

## 元データとサンプリングデータの図示

In [ ]:
# 可視化 (一枚ずつ出力)
plt.figure(figsize=(8, 6))
plt.scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), alpha=0.3, label='Original Data')
plt.title('Original Data Distribution')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-data_2d_plt_lim, data_2d_plt_lim)
plt.ylim(-data_2d_plt_lim, data_2d_plt_lim)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(sampled_data_2d[:, 0], sampled_data_2d[:, 1], alpha=0.3, label='Sampled Data')
plt.title('Sampled Data Distribution')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-sampled_data_plt_lim, sampled_data_plt_lim)
plt.ylim(-sampled_data_plt_lim, sampled_data_plt_lim)
plt.legend()
plt.grid(True)

In [ ]:
# (オプション) KDEによる可視化
def plot_kde_2d(data, ax, title):
    kde = gaussian_kde(data.T)
    x, y = np.mgrid[slice(-5, 5, 0.1), slice(-5, 5, 0.1)]
    positions = np.vstack([x.ravel(), y.ravel()])
    z = np.reshape(kde(positions).T, x.shape)
    ax.imshow(np.rot90(z), cmap=plt.cm.viridis, extent=[-5, 5, -5, 5], aspect='auto')
    ax.scatter(data[:, 0], data[:, 1], s=5, alpha=0.3)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plot_kde_2d(data_2d.numpy(), axes[0], 'Original Data KDE')
plot_kde_2d(sampled_data_2d, axes[1], 'Sampled Data KDE')
plt.tight_layout()
plt.show()